In [1]:
%config InlineBackend.figure_format = 'retina'

In [2]:
from pathlib import Path

import getdist
import matplotlib.pyplot as plt
import numpy as np
from getdist import plots
from scipy.stats import gaussian_kde

from posterior_eigenmodes import (
    add_all_mode_derived_parameters,
    eigenmode_report,
    eigenmodes,
    get_mode_samples,
)
from posterior_eigenmodes.interpretation import (
    format_mode_direction,
    format_mode_summary,
    top_correlation_changes,
    top_eigenvector_alignments,
)


# Quick start (version 1.0.0)

For chains with matching names, the entire analysis can be run as
`results = eigenmodes(chain_root_a, chain_root_b)` followed by
`print(eigenmode_report(results))`. Add `interpretation=True` for brief notes.
The examples below retain explicit parameter mappings because the theta names
differ between the two supplied chains. Reports always display the selected names.

From the terminal: `posterior-eigenmodes chains/A chains/B --params omega_b omega_cdm`.
Run `python -m posterior_eigenmodes --help` for covariance files and mappings.


# Inputs

In [3]:
#Reference chains / covmat = Full CMB
chain_root_a = "./chains/Full/IDE/IDE_FP_BAO_SN"
covmat_a = Path("./chains/Full/IDE/IDE_FP_BAO_SN.covmat")
params_a = ["omega_b", "omega_cdm", "theta_s_1e2", "delta_DMDE"]

#Second chains / covmat = geometric CMB
chain_root_b = "./chains/compressed/IDE/IDE_P3x3_dBAO_SN"
covmat_b = Path("./chains/compressed/IDE/IDE_P3x3_dBAO_SN.covmat")
params_b = ["omega_b", "omega_cdm", "theta_s_100", "delta_DMDE"]

# Results

Use the folloing function to get the results. Here we input both th MCMC chains and covmats (Recommended).

- chains are only to estimate the posterior shifts.
- covmat is used for eigenmodes analysis 

In [4]:
results = eigenmodes(
    covmat_a=covmat_a,
    covmat_b=covmat_b,
    chain_root_a=chain_root_a,
    chain_root_b=chain_root_b,
    params_A=params_a,
    params_B=params_b,
)

./chains/Full/IDE/IDE_FP_BAO_SN.1.txt
./chains/Full/IDE/IDE_FP_BAO_SN.2.txt
./chains/Full/IDE/IDE_FP_BAO_SN.3.txt
./chains/Full/IDE/IDE_FP_BAO_SN.4.txt


Removed 0.3 as burn in
./chains/compressed/IDE/IDE_P3x3_dBAO_SN.1.txt
./chains/compressed/IDE/IDE_P3x3_dBAO_SN.2.txt
./chains/compressed/IDE/IDE_P3x3_dBAO_SN.3.txt
./chains/compressed/IDE/IDE_P3x3_dBAO_SN.4.txt
Removed 0.3 as burn in


In [5]:
print(eigenmode_report(results, precision=4))

A parameters: omega_b, omega_cdm, theta_s_1e2, delta_DMDE
B parameters: omega_b, omega_cdm, theta_s_100, delta_DMDE

A
Reference posterior

- Mode 1
  - lambda = 0.001494
  - sigma = 0.03866
  - direction = -0.9945 delta_DMDE -0.1052 omega_cdm -0.0005 omega_b -0.0004 theta_s_1e2

- Mode 2
  - lambda = 5.341e-07
  - sigma = 0.0007308
  - direction = -0.9928 omega_cdm +0.1050 delta_DMDE +0.0435 omega_b -0.0382 theta_s_1e2

- Mode 3
  - lambda = 7.355e-08
  - sigma = 0.0002712
  - direction = 0.9973 theta_s_1e2 +0.0638 omega_b -0.0352 omega_cdm +0.0033 delta_DMDE

- Mode 4
  - lambda = 1.455e-08
  - sigma = 0.0001206
  - direction = 0.9970 omega_b -0.0622 theta_s_1e2 +0.0455 omega_cdm -0.0053 delta_DMDE

B
Alternative posterior

- Mode 1
  - lambda = 2.461e-05
  - sigma = 0.004961
  - direction = 0.9978 delta_DMDE +0.0659 omega_cdm -0.0095 theta_s_100 -0.0041 omega_b

- Mode 2
  - lambda = 3.057e-07
  - sigma = 0.0005529
  - direction = 0.9919 omega_cdm +0.0931 theta_s_100 -0.0649 delta_D

## Results from Chains only (without covmat)

Here we input only the MCMC chains. The covmat is inferred from the chains

In [6]:
results_mcmc = eigenmodes(
    chain_root_a=chain_root_a,
    chain_root_b=chain_root_b,
    params_A=params_a,
    params_B=params_b,
)

./chains/Full/IDE/IDE_FP_BAO_SN.1.txt
./chains/Full/IDE/IDE_FP_BAO_SN.2.txt


./chains/Full/IDE/IDE_FP_BAO_SN.3.txt


./chains/Full/IDE/IDE_FP_BAO_SN.4.txt


Removed 0.3 as burn in
./chains/compressed/IDE/IDE_P3x3_dBAO_SN.1.txt
./chains/compressed/IDE/IDE_P3x3_dBAO_SN.2.txt
./chains/compressed/IDE/IDE_P3x3_dBAO_SN.3.txt
./chains/compressed/IDE/IDE_P3x3_dBAO_SN.4.txt
Removed 0.3 as burn in


In [7]:
print(eigenmode_report(results_mcmc, precision=4))

A parameters: omega_b, omega_cdm, theta_s_1e2, delta_DMDE
B parameters: omega_b, omega_cdm, theta_s_100, delta_DMDE

A
Reference posterior

- Mode 1
  - lambda = 0.001508
  - sigma = 0.03883
  - direction = -0.9944 delta_DMDE -0.1053 omega_cdm -0.0005 omega_b -0.0004 theta_s_1e2

- Mode 2
  - lambda = 5.324e-07
  - sigma = 0.0007296
  - direction = -0.9927 omega_cdm +0.1052 delta_DMDE +0.0436 omega_b -0.0397 theta_s_1e2

- Mode 3
  - lambda = 7.452e-08
  - sigma = 0.000273
  - direction = 0.9978 theta_s_1e2 +0.0541 omega_b -0.0372 omega_cdm +0.0035 delta_DMDE

- Mode 4
  - lambda = 1.452e-08
  - sigma = 0.0001205
  - direction = 0.9976 omega_b -0.0524 theta_s_1e2 +0.0454 omega_cdm -0.0052 delta_DMDE

B
Alternative posterior

- Mode 1
  - lambda = 2.468e-05
  - sigma = 0.004968
  - direction = 0.9976 delta_DMDE +0.0678 omega_cdm -0.0094 theta_s_100 -0.0046 omega_b

- Mode 2
  - lambda = 3.169e-07
  - sigma = 0.0005629
  - direction = 0.9919 omega_cdm +0.0938 theta_s_100 -0.0668 delta_DM

## Results from covmat only (no MCMC)

Here we input only the covmats. If you want the shift analysis you need to input also vectors with mean values of params or direclty the MCMC chains as above.

In [8]:
# use this function to get the results using the Cobaya covmats.
# if chain roots are also provided, the covmat analysis is kept,
# and the chains are only used to estimate the posterior shifts.
results_covmat = eigenmodes(
    covmat_a=covmat_a,
    covmat_b=covmat_b,
    params_A=params_a,
    params_B=params_b,
)

In [9]:
print(eigenmode_report(results_covmat, precision=4))

A parameters: omega_b, omega_cdm, theta_s_1e2, delta_DMDE
B parameters: omega_b, omega_cdm, theta_s_100, delta_DMDE

A
Reference posterior

- Mode 1
  - lambda = 0.001494
  - sigma = 0.03866
  - direction = -0.9945 delta_DMDE -0.1052 omega_cdm -0.0005 omega_b -0.0004 theta_s_1e2

- Mode 2
  - lambda = 5.341e-07
  - sigma = 0.0007308
  - direction = -0.9928 omega_cdm +0.1050 delta_DMDE +0.0435 omega_b -0.0382 theta_s_1e2

- Mode 3
  - lambda = 7.355e-08
  - sigma = 0.0002712
  - direction = 0.9973 theta_s_1e2 +0.0638 omega_b -0.0352 omega_cdm +0.0033 delta_DMDE

- Mode 4
  - lambda = 1.455e-08
  - sigma = 0.0001206
  - direction = 0.9970 omega_b -0.0622 theta_s_1e2 +0.0455 omega_cdm -0.0053 delta_DMDE

B
Alternative posterior

- Mode 1
  - lambda = 2.461e-05
  - sigma = 0.004961
  - direction = 0.9978 delta_DMDE +0.0659 omega_cdm -0.0095 theta_s_100 -0.0041 omega_b

- Mode 2
  - lambda = 3.057e-07
  - sigma = 0.0005529
  - direction = 0.9919 omega_cdm +0.0931 theta_s_100 -0.0649 delta_D